# Chapter 3: CPU / GPU benchmarks
Select a GPU runtime, then run these cells in order. The `colab_version` branch
must contain the updated files on GitHub before cloning. No source patching is needed.
The setup selects the assigned GPU architecture automatically.
Your original `Colab_version.ipynb` is preserved locally as a reference.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def run_logged(command, check=True):
    """Forward stdout and stderr into this notebook, including tracebacks."""
    print("$", " ".join(map(str, command)), flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=env) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    if check and code:
        raise subprocess.CalledProcessError(code, command)
    return code


repo = Path("/content/CMODE")
if not repo.exists():
    run_logged(["git", "clone", "--branch", "colab_version", "--single-branch",
                "https://github.com/behzadtabari/CMODE.git", str(repo)])
branch = subprocess.check_output(["git", "-C", str(repo), "branch", "--show-current"], text=True).strip()
if branch != "colab_version":
    raise RuntimeError(f"Expected colab_version, found {branch}. Use a fresh runtime or switch the checkout.")
os.chdir(repo)
run_logged(["git", "pull", "--ff-only", "origin", "colab_version"])
run_logged(["git", "log", "-1", "--oneline"])
print("Repository:", repo, "Branch:", branch)


In [ ]:
# Build once. Both build logs and Python errors are displayed below.
import shutil
if shutil.which("nvcc") is None:
    raise RuntimeError("CUDA compiler missing. Select a GPU runtime with the CUDA toolkit.")
run_logged(["nvidia-smi"])
run_logged([sys.executable, "-m", "pip", "install", ".[test,examples]",
            "-Ccmake.define.ODELAB_ENABLE_CUDA=ON",
            "-Ccmake.define.CMAKE_CUDA_ARCHITECTURES=native"])
run_logged([sys.executable, "-c",
    "import odelab; print(odelab.__file__); assert odelab.cuda_available(), 'CUDA device unavailable'"])


In [ ]:
import html
import json
from datetime import datetime, timezone
from IPython.display import HTML, FileLink, display


def display_table(rows, columns):
    table = "<table><thead><tr>" + "".join(
        f"<th>{html.escape(title)}</th>" for _, title, _ in columns) + "</tr></thead><tbody>"
    for row in rows:
        table += "<tr>" + "".join(
            f"<td>{html.escape(format(row[key], fmt))}</td>" for key, _, fmt in columns) + "</tr>"
    display(HTML(table + "</tbody></table>"))


counts, steps, h, repeats = [1, 64, 256], 2000, 0.01, 3
output = Path("build") / datetime.now(timezone.utc).strftime("hopf_%Y%m%d_%H%M%S_%f")
output.mkdir(parents=True, exist_ok=False)
json_path, csv_path = output / "results.json", output / "results.csv"
status = run_logged([
    sys.executable, "-u", "examples/chapter3_exercises/hopfbi_gpu_cpu_benchmarking.py",
    "--require-cuda", "--counts", *map(str, counts), "--steps", str(steps),
    "--h", str(h), "--repeats", str(repeats), "--json", str(json_path), "--csv", str(csv_path),
], check=False)
if json_path.exists():
    results = json.loads(json_path.read_text())
    rows = results["rows"]
    display_table(rows, [
        ("batch", "Batch", "d"), ("backend", "Backend", "s"),
        ("median_ms", "Median ms", ".3f"), ("min_ms", "Minimum ms", ".3f"),
        ("speedup_vs_cpp", "Speedup vs C++", ".2f"),
        ("max_abs_error_ref", "Max error vs reference", ".6e"),
        ("rmse_ref", "RMSE vs reference", ".6e"),
        ("max_abs_diff_cpp", "Max difference vs C++", ".6e"),
    ])
    print("Reference:", results["reference"])
    display(FileLink(str(csv_path)), FileLink(str(json_path)))
if status:
    raise RuntimeError("Hopf benchmark failed; see streamed errors above.")
if not json_path.exists():
    raise RuntimeError("Missing results; check the repository revision and rerun setup.")


## Read the results
The table reports each backend separately for every batch size. The **reference
error** compares forward Euler with a tight-tolerance SciPy DOP853 numerical
solution; it is an estimate, not an error against a known exact solution.
The **difference vs C++** measures agreement between implementations and can be
near machine precision even when Euler's reference error is much larger.
All timings are median/minimum elapsed milliseconds over the chosen repetitions,
following a warm-up. Compilation, reference generation, and plotting are excluded.
The C++ CPU loop calls a Python RHS; CUDA evaluates it on-device. GPU elapsed time
includes transfers, allocation, and completion, rather than only kernel time.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for backend in dict.fromkeys(row["backend"] for row in rows):
    series = [row for row in rows if row["backend"] == backend]
    x = [row["batch"] for row in series]
    axes[0].plot(x, [row["median_ms"] for row in series], "o-", label=backend)
    axes[1].plot(x, [row["max_abs_error_ref"] for row in series], "o--", label=backend)
axes[0].set(xlabel="Batch size", ylabel="Median elapsed time (ms)", yscale="log", title="CPU / GPU timings")
axes[1].set(xlabel="Batch size", ylabel="Maximum absolute reference error", title="Euler error vs DOP853")
for ax in axes:
    ax.set_xscale("log")
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.savefig(output / "benchmark.png", dpi=180)
plt.show()
display(FileLink(str(output / "benchmark.png")))


## Method of lines: compare NumPy, SciPy, and your C++/CUDA library
The spatial discretization of $u_t=u_{xx}$ on the unit rod produces the
**coupled ODE system** $M\dot u=Ku$. Crank–Nicolson is the trapezoidal rule:
$(M-\Delta t K/2)u^{n+1}=(M+\Delta t K/2)u^n$.
Example data: $u(0,t)=u(1,t)=0$, $u(x,0)=\sin(\pi x)$, $\alpha=1$.
The continuum solution $e^{-\pi^2t}\sin(\pi x)$ measures the combined space/time error.

Four backends use the **same matrices, random nodes, time steps, and float64**:
- NumPy arrays and Python Thomas-algorithm loops.
- SciPy sparse LU and a Python time loop.
- Your native C++ Thomas solver with a C++ time loop.
- Your native CUDA parallel cyclic reduction solver, including setup on the GPU.

Every run draws independent uniform interior nodes, without spacing restrictions.
The seed and actual grid are saved. Setup is reused within each solve; each timed
repeat starts fresh. Warm-up and matrix assembly are excluded. Solve time measures
completed time stepping; total time also includes setup, Python conversions and
GPU transfers/cleanup. Speedups and agreement use **your C++ CPU solver** as baseline.
A small single rod may run faster on CPU due to CUDA launch overhead.
Run these cells after package setup; the Hopf cells are optional for this example.


In [ ]:
# Use the compiled odelab library built in the initial setup cell.
run_logged([sys.executable, "-c",
    "import odelab; from odelab import solve_tridiagonal_cn; "
    "print('Native package:', odelab.__file__); "
    "assert odelab.cuda_available(), 'Rebuild odelab with CUDA in a GPU runtime'"])


In [ ]:
import csv
import html
import json
from datetime import datetime, timezone
from IPython.display import HTML, Image, FileLink, display

# Each click of Run generates fresh interior nodes. No fixed seed is passed.
nodes, steps, final_time, repeats = [64, 256, 1024], 200, 0.1, 3
mol_output = Path("build") / datetime.now(timezone.utc).strftime("mol_%Y%m%d_%H%M%S_%f")
status = run_logged([
    sys.executable, "-u", "examples/chapter3_exercises/method_of_line_1D_cpu_gpu.py",
    "--require-cuda", "--nodes", *map(str, nodes), "--steps", str(steps),
    "--final-time", str(final_time), "--repeats", str(repeats), "--output", str(mol_output),
], check=False)
report_path = mol_output / "results.json"
if report_path.exists():
    mol_results = json.loads(report_path.read_text())
    print("Random seed:", mol_results["seed"])
    columns = [
        ("interior_nodes", "Interior nodes", "d"), ("backend", "Backend", "s"),
        ("median_setup_ms", "Setup ms", ".3f"), ("median_solve_ms", "Solve ms", ".3f"),
        ("median_total_ms", "Total ms", ".3f"),
        ("solve_speedup_vs_cpu", "Solve speedup vs C++", ".2f"),
        ("total_speedup_vs_cpu", "Total speedup vs C++", ".2f"),
        ("max_abs_error", "Max exact error", ".6e"),
        ("weighted_l2_error", "Weighted L2 error", ".6e"),
        ("max_abs_diff_cpu", "Difference vs C++", ".6e"),
    ]
    table = "<table><tr>" + "".join(f"<th>{title}</th>" for _,title,_ in columns) + "</tr>"
    for row in mol_results["rows"]:
        table += "<tr>" + "".join(
            f"<td>{html.escape(format(row[key], fmt))}</td>" for key, _, fmt in columns) + "</tr>"
    display(HTML(table + "</table>"))
    display(FileLink(str(mol_output / "results.csv")), FileLink(str(report_path)),
            FileLink(str(mol_output / "grids_and_solutions.npz")))
    if (mol_output / "comparison.png").exists():
        display(Image(filename=str(mol_output / "comparison.png")))
if status:
    raise RuntimeError("Method-of-lines benchmark failed; see streamed output above.")
if not report_path.exists():
    raise RuntimeError("Method-of-lines benchmark produced no results file.")


In [ ]:
run_logged([sys.executable, "-m", "pytest", "tests/python", "-q"])
run_logged([sys.executable, "examples/chapter3_exercises/hopf_bifurcation.py",
                "--no-show", "--save", "build/hopf.png"])
from IPython.display import Image, display
display(Image(filename="build/hopf.png"))
